# Export a BioSTEAM flowsheet to SFF

This tutorial runs a small, genuine [BioSTEAM](https://biosteam.readthedocs.io/) simulation
(one stream through a heat exchanger) and exports the simulated `System` to a
conforming SFF JSON file using the reference exporter shipped in this repo.

A few things worth knowing before the code cell:

- **The versioned exporter is selected by `sff_version`.** `export_biosteam_flowsheet`
  resolves the requested version string (e.g. `"0.1.5"`) to the matching
  `export_biosteam_flowsheet_sff_<major>_<minor>_<patch>` function and writes that
  string into `metadata.sff_version` in the output. `available_sff_versions()` lists
  every version the installed exporter supports.
- **The same call works on any simulated BioSTEAM `System`.** The two-unit toy system
  built below exports through exactly the same `export_biosteam_flowsheet(system, path,
  sff_version=...)` call as a full process model — the exporter only needs a `System`
  that has already been simulated.
- **Whole-model Bioindustrial-Park systems export the same way, but are heavier.** The
  18-flowsheet corpus under `pisces_sff/export/exported_flowsheets/bioindustrial_park/` (see
  the [Read and traverse](read_and_traverse.ipynb) tutorial) was produced from full
  process-scale `Bioindustrial-Park` systems using this same call — they just take much
  longer to simulate than the toy system here.

The export below writes its output JSON to a temporary scratch directory
(`tempfile.mkdtemp()`) rather than into the committed corpus.

In [1]:
import tempfile, os, json
import biosteam as bst
from pisces_sff import export_biosteam_flowsheet, available_sff_versions

print("available SFF versions:", available_sff_versions())

bst.settings.set_thermo(["Water", "Ethanol"])
feed = bst.Stream("feed", Water=1000, Ethanol=100, units="kg/hr", T=298.15)
feed.price = 0.5
H1 = bst.HXutility("H1", ins=feed, outs="hot", T=350)
system = bst.System("small_sys", path=(H1,))
system.simulate()
H1.outs[0].price = 1.0

# The exporter reads metadata.TEA_year off a real bst.TEA object (it is not
# optional in practice despite the keyword's None default) -- these
# construction args are finance placeholders; only tea.duration[0] is read.
tea = bst.TEA(
    system=system, IRR=0.15, duration=(2020, 2030),
    depreciation="MACRS7", income_tax=0.21, operating_days=330.,
    lang_factor=3., construction_schedule=(0.4, 0.6),
    startup_months=0., startup_FOCfrac=0., startup_VOCfrac=0.,
    startup_salesfrac=0., WC_over_FCI=0.05, finance_interest=0.,
    finance_years=0, finance_fraction=0.,
)

out = os.path.join(tempfile.mkdtemp(), "small_sys.sff.json")
export_biosteam_flowsheet(system, out, sff_version="0.1.5", tea=tea)

doc = json.load(open(out))
print("sff_version:", doc["metadata"]["sff_version"])
print("units:", len(doc["units"]), "streams:", len(doc["streams"]))

available SFF versions: ['0.0.5', '0.0.6', '0.0.7', '0.0.8', '0.0.9', '0.0.10', '0.0.11', '0.0.12', '0.1.0', '0.1.1', '0.1.2', '0.1.3', '0.1.4', '0.1.5']
sff_version: 0.1.5
units: 1 streams: 2


The printed `sff_version` confirms which versioned exporter ran, and the
`units`/`streams` counts confirm the graph came through: one `HXutility` unit and
its two connected streams (`feed` in, `hot` out). From here, the resulting file can
be checked with `validate_flowsheet_against_SFF` — see the
[Validate an SFF file](validate_an_sff_file.ipynb) tutorial.